# P-COLD Feature Engineering and Data Preparation

## Overview
This notebook processes the P-COLD operational risk dataset into a structured format suitable for both analytical reporting and machine learning modeling.

It performs data cleaning, feature engineering, aggregation, and target generation to support downstream star schema construction and AI-based risk forecasting.

## Objectives
- Clean and standardize raw P-COLD data
- Engineer loss-based, temporal, and categorical features
- Construct dimension and fact tables for a star schema
- Aggregate data to a monthly modeling grain
- Generate forward-looking targets for multiple forecasting horizons (90, 180, 365 days)

## Key Features Engineered
- Lag features (1, 2, 3, 6, 12 months)
- Rolling aggregates (3, 6, 12 months)
- Loss severity indicators
- Frequency-based encodings
- Time-based features (month, quarter, year)

## Outputs
- Cleaned event-level dataset
- Dimension tables (risk, business line, causal factor, location)
- Fact table for star schema
- Forecasting datasets for 90d, 180d, and 365d horizons

## Notes
This notebook forms the core data pipeline for both the dashboard and the machine learning model.

In [1]:
import os
import warnings
import numpy as np
import pandas as pd

# Configure project folders and modeling assumptions.
# HORIZON_DAYS_LIST controls the forecasting windows exported for model testing.

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# =========================
# PATHS
# =========================
RAW_DIR = "../data/raw/pcold"
INTERIM_DIR = "../data/interim"
PROCESSED_DIR = "../data/processed"
OUTPUT_DIR = "../outputs"

for d in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

PCOLD_FILE = os.path.join(RAW_DIR, "P-COLD-English ver.xlsx")
DICT_FILE = os.path.join(RAW_DIR, "Data dictionary.xlsx")

# =========================
# MODEL / LABEL CONFIG
# =========================
HORIZON_DAYS_LIST = [90, 180, 365]
HIGH_LOSS_THRESHOLD_CNY = 10_000_000
ROLLING_WINDOWS = [3, 6, 12]

print("Configured paths:")
print("PCOLD_FILE:", PCOLD_FILE)
print("DICT_FILE :", DICT_FILE)

Configured paths:
PCOLD_FILE: ../data/raw/pcold\P-COLD-English ver.xlsx
DICT_FILE : ../data/raw/pcold\Data dictionary.xlsx


## Import File

In [2]:
# Load the P-COLD operational risk dataset and its data dictionary.
# The dictionary is kept for reference, while the main dataset drives the pipeline.

df_raw = pd.read_excel(PCOLD_FILE)
dict_sheets = pd.read_excel(DICT_FILE, sheet_name=None)

print("P-COLD shape:", df_raw.shape)
print("P-COLD columns:", df_raw.columns.tolist())
print("\nDictionary sheets:", list(dict_sheets.keys()))

display(df_raw.head())

P-COLD shape: (3725, 12)
P-COLD columns: ['Event ID', 'Occurrence year', 'Occurrence month', 'End year', 'End month', 'Loss amount (Unit: 10,000 CNY)', 'Bank involved code', 'Province occurred', 'City occurred', 'Causal factor', 'Event type', 'Business line']

Dictionary sheets: ['Description of each field', 'Province occured', 'City occurred', 'Causal factor', 'Event type', 'Business line']


,Event ID,Occurrence year,Occurrence month,End year,End month,"Loss amount (Unit: 10,000 CNY)",Bank involved code,Province occurred,City occurred,Causal factor,Event type,Business line
0,1,1999,8,2000,11,10200,"5,6,7,42",Shaanxi,Xi'an,People,Internal fraud,Commercial banking
1,2,1998,12,2000,11,420,4,Guangdong,Dongguan,People,Internal fraud,No business line
2,3,1997,7,2000,4,332,4,Guangdong,Shenzhen,People,Internal fraud,Payment and settlement
3,4,2000,2,F,F,2102,4,Tianjin,F,People,Internal fraud,Payment and settlement
4,5,1995,2,2000,9,1580,4,Beijing,Beijing,People,Internal fraud,No business line


## Cleaning Helper Functions

In [3]:
# Helper functions standardize missing values, parse year/month fields,
# convert loss amounts, build monthly dates, and categorize loss severity.

def normalize_text(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    if x in ["", "nan", "None", "NULL", "F"]:
        return np.nan
    return x

def parse_year(x):
    x = normalize_text(x)
    if pd.isna(x):
        return np.nan
    try:
        return int(float(x))
    except:
        return np.nan

def parse_month(x):
    x = normalize_text(x)
    if pd.isna(x):
        return np.nan

    special_map = {
        "年初": 1,
        "年末": 12,
        "年底": 12
    }
    if x in special_map:
        return special_map[x]

    try:
        m = int(float(x))
        if 1 <= m <= 12:
            return m
        return np.nan
    except:
        return np.nan

def parse_loss_10k_cny(x):
    x = normalize_text(x)
    if pd.isna(x):
        return np.nan
    try:
        return float(x)
    except:
        return np.nan

def month_start_from_parts(year_val, month_val):
    if pd.isna(year_val) or pd.isna(month_val):
        return pd.NaT
    try:
        return pd.Timestamp(year=int(year_val), month=int(month_val), day=1)
    except:
        return pd.NaT

def parse_bank_codes(x):
    x = normalize_text(x)
    if pd.isna(x):
        return []
    parts = [p.strip() for p in str(x).split(",")]
    cleaned = [p for p in parts if p not in ["", "F", "nan", "None"]]
    return cleaned

def severity_bucket(loss_cny):
    if pd.isna(loss_cny):
        return "Unknown"
    if loss_cny < 100_000:
        return "Low"
    elif loss_cny < 1_000_000:
        return "Medium"
    elif loss_cny < 10_000_000:
        return "High"
    else:
        return "Critical"

## Clean and Standardize P-COLD

In [4]:
# Clean and standardize P-COLD into analysis-ready columns.
# This includes date construction, loss conversion from 10,000 CNY units,
# bank-count features, duration features, severity flags, and a structured description.

df = df_raw.copy()

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].apply(normalize_text)

df = df.rename(columns={
    "Event ID": "event_id",
    "Occurrence year": "occurrence_year",
    "Occurrence month": "occurrence_month",
    "End year": "end_year",
    "End month": "end_month",
    "Loss amount (Unit: 10,000 CNY)": "loss_amount_10k_cny",
    "Bank involved code": "bank_involved_code_raw",
    "Province occurred": "province_occurred",
    "City occurred": "city_occurred",
    "Causal factor": "causal_factor",
    "Event type": "event_type",
    "Business line": "business_line"
})

df["occurrence_year"] = df["occurrence_year"].apply(parse_year)
df["occurrence_month"] = df["occurrence_month"].apply(parse_month)
df["end_year"] = df["end_year"].apply(parse_year)
df["end_month"] = df["end_month"].apply(parse_month)
df["loss_amount_10k_cny"] = df["loss_amount_10k_cny"].apply(parse_loss_10k_cny)

df["event_start_date"] = df.apply(
    lambda r: month_start_from_parts(r["occurrence_year"], r["occurrence_month"]), axis=1
)
df["event_end_date"] = df.apply(
    lambda r: month_start_from_parts(r["end_year"], r["end_month"]), axis=1
)

df["event_id"] = df["event_id"].astype(str).str.strip()

df["gross_loss_cny"] = df["loss_amount_10k_cny"] * 10000
df["recovery_amount_cny"] = 0.0
df["net_loss_cny"] = df["gross_loss_cny"] - df["recovery_amount_cny"]

df["bank_code_list"] = df["bank_involved_code_raw"].apply(parse_bank_codes)
df["num_banks_involved"] = df["bank_code_list"].apply(len)
df["primary_bank_code"] = df["bank_code_list"].apply(lambda x: x[0] if len(x) > 0 else np.nan)

df["event_duration_months"] = (
    (df["end_year"] - df["occurrence_year"]) * 12
    + (df["end_month"] - df["occurrence_month"])
)
df["event_duration_months"] = df["event_duration_months"].where(df["event_duration_months"] >= 0, np.nan)

df["event_year"] = df["event_start_date"].dt.year
df["event_month"] = df["event_start_date"].dt.month
df["event_quarter"] = df["event_start_date"].dt.quarter
df["year_month"] = df["event_start_date"].dt.to_period("M").astype(str)

df["loss_severity_bucket"] = df["gross_loss_cny"].apply(severity_bucket)
df["is_high_loss"] = (df["gross_loss_cny"] >= HIGH_LOSS_THRESHOLD_CNY).astype(int)
df["log_gross_loss"] = np.log1p(df["gross_loss_cny"])

df["description"] = (
    "Operational risk event: "
    + df["event_type"].fillna("Unknown type").astype(str)
    + " caused by "
    + df["causal_factor"].fillna("Unknown cause").astype(str)
    + " in "
    + df["business_line"].fillna("Unknown business line").astype(str)
    + " located in "
    + df["province_occurred"].fillna("Unknown province").astype(str)
    + ", "
    + df["city_occurred"].fillna("Unknown city").astype(str)
    + "."
)

print("Cleaned shape:", df.shape)
display(df.head())

Cleaned shape: (3725, 29)


,event_id,occurrence_year,occurrence_month,end_year,end_month,loss_amount_10k_cny,bank_involved_code_raw,province_occurred,city_occurred,causal_factor,event_type,business_line,event_start_date,event_end_date,gross_loss_cny,recovery_amount_cny,net_loss_cny,bank_code_list,num_banks_involved,primary_bank_code,event_duration_months,event_year,event_month,event_quarter,year_month,loss_severity_bucket,is_high_loss,log_gross_loss,description
0,1,1999.0,8.0,2000.0,11.0,10200.0,"5,6,7,42",Shaanxi,Xi'an,People,Internal fraud,Commercial banking,1999-08-01,2000-11-01,102000000.0,0.0,102000000.0,"[5, 6, 7, 42]",4,5,15.0,1999.0,8.0,3.0,1999-08,Critical,1,18.440483,Operational risk event: Internal fraud caused ...
1,2,1998.0,12.0,2000.0,11.0,420.0,4,Guangdong,Dongguan,People,Internal fraud,No business line,1998-12-01,2000-11-01,4200000.0,0.0,4200000.0,[4],1,4,23.0,1998.0,12.0,4.0,1998-12,High,0,15.250595,Operational risk event: Internal fraud caused ...
2,3,1997.0,7.0,2000.0,4.0,332.0,4,Guangdong,Shenzhen,People,Internal fraud,Payment and settlement,1997-07-01,2000-04-01,3320000.0,0.0,3320000.0,[4],1,4,33.0,1997.0,7.0,3.0,1997-07,High,0,15.015476,Operational risk event: Internal fraud caused ...
3,4,2000.0,2.0,NaN,NaN,2102.0,4,Tianjin,NaN,People,Internal fraud,Payment and settlement,2000-02-01,NaT,21020000.0,0.0,21020000.0,[4],1,4,NaN,2000.0,2.0,1.0,2000-02,Critical,1,16.860985,Operational risk event: Internal fraud caused ...
4,5,1995.0,2.0,2000.0,9.0,1580.0,4,Beijing,Beijing,People,Internal fraud,No business line,1995-02-01,2000-09-01,15800000.0,0.0,15800000.0,[4],1,4,67.0,1995.0,2.0,1.0,1995-02,Critical,1,16.575521,Operational risk event: Internal fraud caused ...


## Data Quality Check

In [5]:
# Review missingness, category distributions, and loss statistics
# to validate that the cleaned dataset is usable for modeling and dashboarding.

print("Null summary:")
display(df.isna().mean().sort_values(ascending=False).to_frame("null_pct").head(20))

print("\nTop event types:")
display(df["event_type"].value_counts(dropna=False).head(10).to_frame("count"))

print("\nTop business lines:")
display(df["business_line"].value_counts(dropna=False).head(10).to_frame("count"))

print("\nTop causal factors:")
display(df["causal_factor"].value_counts(dropna=False).head(10).to_frame("count"))

print("\nLoss stats (CNY):")
display(df["gross_loss_cny"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame())

Null summary:


,null_pct
event_duration_months,0.620940
event_end_date,0.561342
end_month,0.561074
end_year,0.429530
bank_involved_code_raw,0.272215
primary_bank_code,0.272215
city_occurred,0.234094
province_occurred,0.194094
event_start_date,0.173423
event_year,0.173423



Top event types:


,count
event_type,
Internal fraud,1373
External fraud,1109
"Execution, delivery & process management",608
"Clients, products & business practices",544
Business disruption and system failures,33
Damage to physical assets,31
Employment practices and workplace safety,25
NaN,2



Top business lines:


,count
business_line,
Retail banking,1476
Asset management,825
Commercial banking,682
Payment and settlement,349
No business line,217
Trading & sales,138
Agency services,23
Corporate finance,10
Retail brokerage,3



Top causal factors:


,count
causal_factor,
People,1688
External event,1177
Process,800
System,58
NaN,2



Loss stats (CNY):


,gross_loss_cny
count,3.722000e+03
mean,2.849058e+08
std,7.042362e+09
min,5.000000e-02
50%,7.000000e+05
75%,1.212393e+07
90%,1.300000e+08
95%,4.000000e+08
99%,2.742320e+09
max,4.100000e+11


## Build Event Table

In [6]:
# Create a cleaned event-level table that preserves the original P-COLD event grain
# while adding engineered fields used later for the star schema and ML features.

pcold_clean_events = df[[
    "event_id",
    "event_start_date",
    "event_end_date",
    "occurrence_year",
    "occurrence_month",
    "end_year",
    "end_month",
    "event_year",
    "event_month",
    "event_quarter",
    "year_month",
    "loss_amount_10k_cny",
    "gross_loss_cny",
    "recovery_amount_cny",
    "net_loss_cny",
    "log_gross_loss",
    "loss_severity_bucket",
    "is_high_loss",
    "bank_involved_code_raw",
    "num_banks_involved",
    "primary_bank_code",
    "province_occurred",
    "city_occurred",
    "causal_factor",
    "event_type",
    "business_line",
    "event_duration_months",
    "description"
]].copy()

pcold_clean_events.to_csv(os.path.join(INTERIM_DIR, "pcold_clean_events.csv"), index=False)

print("Saved:", os.path.join(INTERIM_DIR, "pcold_clean_events.csv"))
display(pcold_clean_events.head())

Saved: ../data/interim\pcold_clean_events.csv


,event_id,event_start_date,event_end_date,occurrence_year,occurrence_month,end_year,end_month,event_year,event_month,event_quarter,year_month,loss_amount_10k_cny,gross_loss_cny,recovery_amount_cny,net_loss_cny,log_gross_loss,loss_severity_bucket,is_high_loss,bank_involved_code_raw,num_banks_involved,primary_bank_code,province_occurred,city_occurred,causal_factor,event_type,business_line,event_duration_months,description
0,1,1999-08-01,2000-11-01,1999.0,8.0,2000.0,11.0,1999.0,8.0,3.0,1999-08,10200.0,102000000.0,0.0,102000000.0,18.440483,Critical,1,"5,6,7,42",4,5,Shaanxi,Xi'an,People,Internal fraud,Commercial banking,15.0,Operational risk event: Internal fraud caused ...
1,2,1998-12-01,2000-11-01,1998.0,12.0,2000.0,11.0,1998.0,12.0,4.0,1998-12,420.0,4200000.0,0.0,4200000.0,15.250595,High,0,4,1,4,Guangdong,Dongguan,People,Internal fraud,No business line,23.0,Operational risk event: Internal fraud caused ...
2,3,1997-07-01,2000-04-01,1997.0,7.0,2000.0,4.0,1997.0,7.0,3.0,1997-07,332.0,3320000.0,0.0,3320000.0,15.015476,High,0,4,1,4,Guangdong,Shenzhen,People,Internal fraud,Payment and settlement,33.0,Operational risk event: Internal fraud caused ...
3,4,2000-02-01,NaT,2000.0,2.0,NaN,NaN,2000.0,2.0,1.0,2000-02,2102.0,21020000.0,0.0,21020000.0,16.860985,Critical,1,4,1,4,Tianjin,NaN,People,Internal fraud,Payment and settlement,NaN,Operational risk event: Internal fraud caused ...
4,5,1995-02-01,2000-09-01,1995.0,2.0,2000.0,9.0,1995.0,2.0,1.0,1995-02,1580.0,15800000.0,0.0,15800000.0,16.575521,Critical,1,4,1,4,Beijing,Beijing,People,Internal fraud,No business line,67.0,Operational risk event: Internal fraud caused ...


## Build Dimensions

In [7]:
# Build dimension tables for risk, business line, causal factor, and location.
# A risk_id is generated from event type, causal factor, and business line.

risk_key = (
    pcold_clean_events["event_type"].fillna("Unknown").astype(str) + " | " +
    pcold_clean_events["causal_factor"].fillna("Unknown").astype(str) + " | " +
    pcold_clean_events["business_line"].fillna("Unknown").astype(str)
)

pcold_clean_events["risk_key"] = risk_key

risk_map = (
    pd.DataFrame({"risk_key": risk_key.unique()})
    .reset_index()
    .rename(columns={"index": "risk_num"})
)
risk_map["risk_id"] = "RISK-" + (risk_map["risk_num"] + 1).astype(str).str.zfill(4)

pcold_clean_events = pcold_clean_events.merge(
    risk_map[["risk_key", "risk_id"]],
    on="risk_key",
    how="left"
)

dim_risk = (
    pcold_clean_events[["risk_id", "event_type", "causal_factor", "business_line", "description"]]
    .drop_duplicates(subset=["risk_id"])
    .rename(columns={
        "event_type": "risk_event_type",
        "causal_factor": "risk_category",
        "business_line": "risk_business_line",
        "description": "risk_description"
    })
    .reset_index(drop=True)
)

dim_risk["risk_owner"] = "Unknown"

dim_business_line = (
    pcold_clean_events[["business_line"]]
    .drop_duplicates()
    .dropna()
    .sort_values("business_line")
    .reset_index(drop=True)
)
dim_business_line["business_line_id"] = "BL-" + (dim_business_line.index + 1).astype(str).str.zfill(3)

dim_causal_factor = (
    pcold_clean_events[["causal_factor"]]
    .drop_duplicates()
    .dropna()
    .sort_values("causal_factor")
    .reset_index(drop=True)
)
dim_causal_factor["causal_factor_id"] = "CF-" + (dim_causal_factor.index + 1).astype(str).str.zfill(3)

dim_location = (
    pcold_clean_events[["province_occurred", "city_occurred"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
dim_location["location_id"] = "LOC-" + (dim_location.index + 1).astype(str).str.zfill(4)

dim_risk.to_csv(os.path.join(PROCESSED_DIR, "dim_risk.csv"), index=False)
dim_business_line.to_csv(os.path.join(PROCESSED_DIR, "dim_business_line.csv"), index=False)
dim_causal_factor.to_csv(os.path.join(PROCESSED_DIR, "dim_causal_factor.csv"), index=False)
dim_location.to_csv(os.path.join(PROCESSED_DIR, "dim_location.csv"), index=False)

print("Saved dimensions to", PROCESSED_DIR)
display(dim_risk.head())

Saved dimensions to ../data/processed


,risk_id,risk_event_type,risk_category,risk_business_line,risk_description,risk_owner
0,RISK-0001,Internal fraud,People,Commercial banking,Operational risk event: Internal fraud caused ...,Unknown
1,RISK-0002,Internal fraud,People,No business line,Operational risk event: Internal fraud caused ...,Unknown
2,RISK-0003,Internal fraud,People,Payment and settlement,Operational risk event: Internal fraud caused ...,Unknown
3,RISK-0004,"Clients, products & business practices",People,Retail banking,"Operational risk event: Clients, products & bu...",Unknown
4,RISK-0005,Internal fraud,People,Retail banking,Operational risk event: Internal fraud caused ...,Unknown


## Build Fact Table

In [8]:
# Build the operational risk fact table by joining event records to dimension IDs.
# Compatibility aliases are added so the downstream star schema notebook can reuse
# the expected field names from the earlier synthetic pipeline.

fact_risk_events = pcold_clean_events.merge(
    dim_location,
    on=["province_occurred", "city_occurred"],
    how="left"
)

fact_risk_events = fact_risk_events.merge(
    dim_business_line,
    on="business_line",
    how="left"
)

fact_risk_events = fact_risk_events.merge(
    dim_causal_factor,
    on="causal_factor",
    how="left"
)

fact_risk_events = fact_risk_events[[
    "event_id",
    "event_start_date",
    "event_end_date",
    "risk_id",
    "business_line",
    "business_line_id",
    "causal_factor",
    "causal_factor_id",
    "province_occurred",
    "city_occurred",
    "location_id",
    "gross_loss_cny",
    "recovery_amount_cny",
    "net_loss_cny",
    "log_gross_loss",
    "loss_severity_bucket",
    "is_high_loss",
    "num_banks_involved",
    "primary_bank_code",
    "event_duration_months",
    "description"
]].copy()

fact_risk_events["event_date"] = fact_risk_events["event_start_date"]
fact_risk_events["gross_loss"] = fact_risk_events["gross_loss_cny"]
fact_risk_events["recovery_amount"] = fact_risk_events["recovery_amount_cny"]
fact_risk_events["net_loss"] = fact_risk_events["net_loss_cny"]
fact_risk_events["business_unit"] = fact_risk_events["business_line"]
fact_risk_events["mitigation_status"] = "Unknown"
fact_risk_events["mitigation_due_date"] = pd.NaT
fact_risk_events["expected_loss"] = fact_risk_events["net_loss_cny"]
fact_risk_events["probability_score"] = np.nan

fact_risk_events.to_csv(os.path.join(PROCESSED_DIR, "fact_risk_events.csv"), index=False)

print("Saved:", os.path.join(PROCESSED_DIR, "fact_risk_events.csv"))
display(fact_risk_events.head())

Saved: ../data/processed\fact_risk_events.csv


,event_id,event_start_date,event_end_date,risk_id,business_line,business_line_id,causal_factor,causal_factor_id,province_occurred,city_occurred,location_id,gross_loss_cny,recovery_amount_cny,net_loss_cny,log_gross_loss,loss_severity_bucket,is_high_loss,num_banks_involved,primary_bank_code,event_duration_months,description,event_date,gross_loss,recovery_amount,net_loss,business_unit,mitigation_status,mitigation_due_date,expected_loss,probability_score
0,1,1999-08-01,2000-11-01,RISK-0001,Commercial banking,BL-003,People,CF-002,Shaanxi,Xi'an,LOC-0001,102000000.0,0.0,102000000.0,18.440483,Critical,1,4,5,15.0,Operational risk event: Internal fraud caused ...,1999-08-01,102000000.0,0.0,102000000.0,Commercial banking,Unknown,NaT,102000000.0,NaN
1,2,1998-12-01,2000-11-01,RISK-0002,No business line,BL-005,People,CF-002,Guangdong,Dongguan,LOC-0002,4200000.0,0.0,4200000.0,15.250595,High,0,1,4,23.0,Operational risk event: Internal fraud caused ...,1998-12-01,4200000.0,0.0,4200000.0,No business line,Unknown,NaT,4200000.0,NaN
2,3,1997-07-01,2000-04-01,RISK-0003,Payment and settlement,BL-006,People,CF-002,Guangdong,Shenzhen,LOC-0003,3320000.0,0.0,3320000.0,15.015476,High,0,1,4,33.0,Operational risk event: Internal fraud caused ...,1997-07-01,3320000.0,0.0,3320000.0,Payment and settlement,Unknown,NaT,3320000.0,NaN
3,4,2000-02-01,NaT,RISK-0003,Payment and settlement,BL-006,People,CF-002,Tianjin,NaN,LOC-0004,21020000.0,0.0,21020000.0,16.860985,Critical,1,1,4,NaN,Operational risk event: Internal fraud caused ...,2000-02-01,21020000.0,0.0,21020000.0,Payment and settlement,Unknown,NaT,21020000.0,NaN
4,5,1995-02-01,2000-09-01,RISK-0002,No business line,BL-005,People,CF-002,Beijing,Beijing,LOC-0005,15800000.0,0.0,15800000.0,16.575521,Critical,1,1,4,67.0,Operational risk event: Internal fraud caused ...,1995-02-01,15800000.0,0.0,15800000.0,No business line,Unknown,NaT,15800000.0,NaN


## Build Monthly Feature Store
For future risk modeling

In [9]:
# Aggregate events to a monthly modeling grain by business line, event type,
# causal factor, and province. Lag and rolling features capture recent risk patterns
# without using future information.

monthly = (
    pcold_clean_events
    .dropna(subset=["event_start_date"])
    .groupby(["business_line", "event_type", "causal_factor", "province_occurred", "year_month"], as_index=False)
    .agg(
        event_count=("event_id", "count"),
        total_loss_cny=("gross_loss_cny", "sum"),
        avg_loss_cny=("gross_loss_cny", "mean"),
        median_loss_cny=("gross_loss_cny", "median"),
        max_loss_cny=("gross_loss_cny", "max"),
        high_loss_count=("is_high_loss", "sum"),
        avg_banks_involved=("num_banks_involved", "mean")
    )
)

monthly["month_start"] = pd.to_datetime(monthly["year_month"] + "-01")
monthly = monthly.sort_values(
    ["business_line", "event_type", "causal_factor", "province_occurred", "month_start"]
).reset_index(drop=True)

group_cols = ["business_line", "event_type", "causal_factor", "province_occurred"]

for lag in [1, 2, 3, 6, 12]:
    monthly[f"event_count_lag_{lag}"] = monthly.groupby(group_cols)["event_count"].shift(lag)
    monthly[f"total_loss_lag_{lag}"] = monthly.groupby(group_cols)["total_loss_cny"].shift(lag)
    monthly[f"high_loss_count_lag_{lag}"] = monthly.groupby(group_cols)["high_loss_count"].shift(lag)

for w in ROLLING_WINDOWS:
    monthly[f"event_count_roll_{w}"] = (
        monthly.groupby(group_cols)["event_count"]
        .transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum())
    )
    monthly[f"total_loss_roll_{w}"] = (
        monthly.groupby(group_cols)["total_loss_cny"]
        .transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum())
    )
    monthly[f"high_loss_roll_{w}"] = (
        monthly.groupby(group_cols)["high_loss_count"]
        .transform(lambda s: s.shift(1).rolling(w, min_periods=1).sum())
    )

monthly["loss_per_event"] = monthly["total_loss_cny"] / monthly["event_count"].replace(0, np.nan)
monthly["log_total_loss"] = np.log1p(monthly["total_loss_cny"])

monthly.to_csv(os.path.join(INTERIM_DIR, "pcold_monthly_features.csv"), index=False)

print("Saved:", os.path.join(INTERIM_DIR, "pcold_monthly_features.csv"))
display(monthly.head())

Saved: ../data/interim\pcold_monthly_features.csv


,business_line,event_type,causal_factor,province_occurred,year_month,event_count,total_loss_cny,avg_loss_cny,median_loss_cny,max_loss_cny,high_loss_count,avg_banks_involved,month_start,event_count_lag_1,total_loss_lag_1,high_loss_count_lag_1,event_count_lag_2,total_loss_lag_2,high_loss_count_lag_2,event_count_lag_3,total_loss_lag_3,high_loss_count_lag_3,event_count_lag_6,total_loss_lag_6,high_loss_count_lag_6,event_count_lag_12,total_loss_lag_12,high_loss_count_lag_12,event_count_roll_3,total_loss_roll_3,high_loss_roll_3,event_count_roll_6,total_loss_roll_6,high_loss_roll_6,event_count_roll_12,total_loss_roll_12,high_loss_roll_12,loss_per_event,log_total_loss
0,Agency services,Business disruption and system failures,System,Guangdong,2008-02,1,0.0,NaN,NaN,NaN,0,1.0,2008-02-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.000000
1,Agency services,"Clients, products & business practices",People,Beijing,2015-05,1,70000.0,70000.0,70000.0,70000.0,0,1.0,2015-05-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70000.0,11.156265
2,Agency services,"Clients, products & business practices",People,Hainan,2009-10,1,50000.0,50000.0,50000.0,50000.0,0,1.0,2009-10-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,50000.0,10.819798
3,Agency services,"Execution, delivery & process management",External event,Shanghai,2010-08,1,6039.0,6039.0,6039.0,6039.0,0,0.0,2010-08-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6039.0,8.706159
4,Agency services,"Execution, delivery & process management",People,Hebei,2003-07,1,1500.0,1500.0,1500.0,1500.0,0,0.0,2003-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1500.0,7.313887


## Forward-looking Labels

In [10]:
# Generate forward-looking labels for each forecast horizon.
# The target indicates whether a high-loss event occurs within the future window.
# Separate datasets are exported for 90, 180, and 365-day horizons.

all_future_datasets = []

for horizon_days in HORIZON_DAYS_LIST:
    future_ds = monthly.copy()
    future_ds["future_window_end"] = future_ds["month_start"] + pd.Timedelta(days=horizon_days)

    def future_label_high_loss(row, base_df):
        same_group = base_df[
            (base_df["business_line"] == row["business_line"]) &
            (base_df["event_type"] == row["event_type"]) &
            (base_df["causal_factor"] == row["causal_factor"]) &
            (base_df["province_occurred"] == row["province_occurred"]) &
            (base_df["month_start"] > row["month_start"]) &
            (base_df["month_start"] <= row["future_window_end"])
        ]
        return int((same_group["high_loss_count"] > 0).any())

    def future_label_any_event(row, base_df):
        same_group = base_df[
            (base_df["business_line"] == row["business_line"]) &
            (base_df["event_type"] == row["event_type"]) &
            (base_df["causal_factor"] == row["causal_factor"]) &
            (base_df["province_occurred"] == row["province_occurred"]) &
            (base_df["month_start"] > row["month_start"]) &
            (base_df["month_start"] <= row["future_window_end"])
        ]
        return int((same_group["event_count"] > 0).any())

    future_ds["future_high_loss"] = future_ds.apply(lambda r: future_label_high_loss(r, monthly), axis=1)
    future_ds["future_any_event"] = future_ds.apply(lambda r: future_label_any_event(r, monthly), axis=1)

    for col in ["business_line", "event_type", "causal_factor", "province_occurred"]:
        freq_map = future_ds[col].value_counts(normalize=True)
        future_ds[f"{col}_freq"] = future_ds[col].map(freq_map)

    future_ds["month_num"] = future_ds["month_start"].dt.month
    future_ds["quarter_num"] = future_ds["month_start"].dt.quarter
    future_ds["year_num"] = future_ds["month_start"].dt.year
    future_ds["horizon_days"] = horizon_days

    num_cols = future_ds.select_dtypes(include=[np.number]).columns.tolist()
    future_ds[num_cols] = future_ds[num_cols].fillna(0)

    out_file = os.path.join(PROCESSED_DIR, f"future_model_dataset_{horizon_days}d.csv")
    future_ds.to_csv(out_file, index=False)
    print("Saved:", out_file)

    all_future_datasets.append(future_ds.copy())

future_model_dataset_all = pd.concat(all_future_datasets, ignore_index=True)
future_model_dataset_all.to_csv(os.path.join(PROCESSED_DIR, "future_model_dataset_all_horizons.csv"), index=False)

print("Saved:", os.path.join(PROCESSED_DIR, "future_model_dataset_all_horizons.csv"))
display(future_model_dataset_all.head())

Saved: ../data/processed\future_model_dataset_90d.csv
Saved: ../data/processed\future_model_dataset_180d.csv
Saved: ../data/processed\future_model_dataset_365d.csv
Saved: ../data/processed\future_model_dataset_all_horizons.csv


,business_line,event_type,causal_factor,province_occurred,year_month,event_count,total_loss_cny,avg_loss_cny,median_loss_cny,max_loss_cny,high_loss_count,avg_banks_involved,month_start,event_count_lag_1,total_loss_lag_1,high_loss_count_lag_1,event_count_lag_2,total_loss_lag_2,high_loss_count_lag_2,event_count_lag_3,total_loss_lag_3,high_loss_count_lag_3,event_count_lag_6,total_loss_lag_6,high_loss_count_lag_6,event_count_lag_12,total_loss_lag_12,high_loss_count_lag_12,event_count_roll_3,total_loss_roll_3,high_loss_roll_3,event_count_roll_6,total_loss_roll_6,high_loss_roll_6,event_count_roll_12,total_loss_roll_12,high_loss_roll_12,loss_per_event,log_total_loss,future_window_end,future_high_loss,future_any_event,business_line_freq,event_type_freq,causal_factor_freq,province_occurred_freq,month_num,quarter_num,year_num,horizon_days
0,Agency services,Business disruption and system failures,System,Guangdong,2008-02,1,0.0,0.0,0.0,0.0,0,1.0,2008-02-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,2008-05-01,0,0,0.00627,0.009404,0.017465,0.044783,2,1,2008,90
1,Agency services,"Clients, products & business practices",People,Beijing,2015-05,1,70000.0,70000.0,70000.0,70000.0,0,1.0,2015-05-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,70000.0,11.156265,2015-07-30,0,0,0.00627,0.142409,0.459472,0.094044,5,2,2015,90
2,Agency services,"Clients, products & business practices",People,Hainan,2009-10,1,50000.0,50000.0,50000.0,50000.0,0,1.0,2009-10-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50000.0,10.819798,2009-12-30,0,0,0.00627,0.142409,0.459472,0.010300,10,4,2009,90
3,Agency services,"Execution, delivery & process management",External event,Shanghai,2010-08,1,6039.0,6039.0,6039.0,6039.0,0,0.0,2010-08-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6039.0,8.706159,2010-10-30,0,0,0.00627,0.152709,0.322884,0.045231,8,3,2010,90
4,Agency services,"Execution, delivery & process management",People,Hebei,2003-07,1,1500.0,1500.0,1500.0,1500.0,0,0.0,2003-07-01,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1500.0,7.313887,2003-09-29,0,0,0.00627,0.152709,0.459472,0.011196,7,3,2003,90


## Final Export

In [11]:
# Print all exported files and final table shapes for auditability.

print("========== FILES WRITTEN ==========")
print(os.path.join(INTERIM_DIR, "pcold_clean_events.csv"))
print(os.path.join(INTERIM_DIR, "pcold_monthly_features.csv"))
print(os.path.join(INTERIM_DIR, "pcold_feature_store.csv"))
print(os.path.join(PROCESSED_DIR, "dim_risk.csv"))
print(os.path.join(PROCESSED_DIR, "dim_business_line.csv"))
print(os.path.join(PROCESSED_DIR, "dim_causal_factor.csv"))
print(os.path.join(PROCESSED_DIR, "dim_location.csv"))
print(os.path.join(PROCESSED_DIR, "fact_risk_events.csv"))
print(os.path.join(PROCESSED_DIR, "future_model_dataset_90d.csv"))
print(os.path.join(PROCESSED_DIR, "future_model_dataset_180d.csv"))
print(os.path.join(PROCESSED_DIR, "future_model_dataset_365d.csv"))
print(os.path.join(PROCESSED_DIR, "future_model_dataset_all_horizons.csv"))

print("\nShapes:")
print("pcold_clean_events      :", pcold_clean_events.shape)
print("dim_risk                :", dim_risk.shape)
print("fact_risk_events        :", fact_risk_events.shape)
print("future_model_dataset_90d:", future_ds.shape)

========== FILES WRITTEN ==========
../data/interim\pcold_clean_events.csv
../data/interim\pcold_monthly_features.csv
../data/interim\pcold_feature_store.csv
../data/processed\dim_risk.csv
../data/processed\dim_business_line.csv
../data/processed\dim_causal_factor.csv
../data/processed\dim_location.csv
../data/processed\fact_risk_events.csv
../data/processed\future_model_dataset_90d.csv
../data/processed\future_model_dataset_180d.csv
../data/processed\future_model_dataset_365d.csv
../data/processed\future_model_dataset_all_horizons.csv

Shapes:
pcold_clean_events      : (3725, 30)
dim_risk                : (98, 6)
fact_risk_events        : (3725, 30)
future_model_dataset_90d: (2233, 50)
